In [2]:
import numpy as np
import pymc as pm
import arviz as az
# -- use this line at the beginning of your notebook to turn on interactive plots
%matplotlib notebook
import scipy.io
# note these imports, thay may be useful in your own task
import statsmodels.api as sm

import matplotlib.pyplot as plt  # plotting library
import numpy as np  # work with numeric arrays without labeled axes
import xarray as xr  # work with arrays with labeled axes
import pandas as pd

In [3]:

def calculate_condition_index(df):
    # Step 1: Compute the correlation matrix
    corr_matrix = df.corr()

    # Step 2: Compute eigenvalues of the correlation matrix
    eigenvalues, _ = np.linalg.eig(corr_matrix)

    # Step 3: Sort eigenvalues in descending order (for clarity)
    sorted_eigenvalues = np.sort(eigenvalues)[::-1]

    # Step 4: Compute condition index as sqrt(largest_eigenvalue / current_eigenvalue)
    max_eigenvalue = sorted_eigenvalues[0]
    condition_indices = [np.sqrt(max_eigenvalue / ev) for ev in sorted_eigenvalues]
    
    # Step 5: Find the maximum Condition Index
    max_condition_index = max(condition_indices)
    
    # Step 5: Return condition indices as a DataFrame
    condition_index_df = pd.DataFrame({
        'Eigenvalue': sorted_eigenvalues,
        'Condition Index': condition_indices
    })

    return condition_index_df, max_condition_index
def calculate_vif1(df):
    # Initialize a DataFrame to store the VIF for each variable
    vif_data = pd.DataFrame()
    vif_data["Variable"] = df.columns
    vif_values = []

    for i in range(df.shape[1]):
        # Take the i-th column as the dependent variable, and the remaining columns as independent variables
        y = df.iloc[:, i]
        X = df.drop(df.columns[i], axis=1)

        # Calculate the R^2 value: beta = (X^T X)^(-1) X^T y
        X = np.column_stack([np.ones(X.shape[0]), X])  # Add an intercept term
        beta = np.linalg.lstsq(X, y, rcond=None)[0]  # Solve for the least squares solution
        y_pred = X @ beta  # Predicted values
        r_squared = 1 - np.sum((y - y_pred) ** 2) / np.sum((y - y.mean()) ** 2)

        # Calculate the VIF value
        vif = 1 / (1 - r_squared)
        vif_values.append(vif)

    vif_data["VIF"] = vif_values
    return vif_data

def calculate_variance_decomposition(df):
    # Step 1: Calculate the covariance matrix
    cov_matrix = df.cov()

    # Step 2: Calculate eigenvalues and eigenvectors of the covariance matrix
    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

    # Step 3: Compute the variance decomposition proportions for each variable
    variance_decomposition = []
    for i in range(len(df.columns)):
        # Compute the variance proportion of the i-th variable for each eigenvalue
        proportions = (eigenvectors[i, :] ** 2) * eigenvalues / cov_matrix.iloc[i, i]
        variance_decomposition.append(proportions / proportions.sum())

    # Convert results to a DataFrame
    variance_decomposition_df = pd.DataFrame(
        variance_decomposition, 
        columns=[f'Eigenvalue_{j+1}' for j in range(len(eigenvalues))],
        index=df.columns
    )
    return variance_decomposition_df

def calculate_determinant_of_correlation_matrix(df):
    # Step 1: Compute the correlation matrix
    corr_matrix = df.corr()

    # Step 2: Compute eigenvalues of the correlation matrix
    eigenvalues = np.linalg.eigvals(corr_matrix)

    # Step 3: Calculate the determinant as the product of the eigenvalues
    determinant = np.prod(eigenvalues)

    return determinant

In [6]:
file_path = 'LH_metal_DB_v1.12.csv'  # 替换为你的文件路径
df = pd.read_csv(file_path)
df.loc[:, 'TOK'] = df['TOK'].str.strip()
df['IP']=df['IP'].abs()/1000000
df['BT']=df['BT'].abs()
df['PLTH']=df['PLTH']/1000000
df['NEL']=df['NEL']/1e19


# 指定你想在新 DataFrame 中包含的列
columns_to_include = ['TOK', 'SPLASMA','BT','IP','RGEO','PLTH','NEL','PGASA',"DIVCON"]
new_df = df[df['SELEC2024'] == 1][columns_to_include].copy()
mu0 = 4 * np.pi * 1e-7

new_df['BP']=1000000*(mu0 * new_df['IP']) / (new_df['SPLASMA']/(2 * np.pi * new_df['RGEO']))

new_df['PLTH/S/2']=new_df['PLTH']/new_df['SPLASMA']/2


columns_to_include_for_regre = ['PLTH/S/2','BT','BP','IP','NEL','PGASA']
unique_divcon = new_df['DIVCON'].unique()

# 2. 向原始 DataFrame 添加新列
for divcon_type in unique_divcon:
    new_df[divcon_type] = (new_df['DIVCON'] == divcon_type).astype(int)
multi_df = new_df[columns_to_include_for_regre].copy()
multi_df = multi_df.apply(np.log)
new_columns = ['$P_{LH,norm}$',"$B_T$", "$B_P$", "$I_P$","$n_{e,line}$","$M_{eff}$"]

# 修改列名
multi_df.columns = new_columns
correlation_matrix = multi_df.corr()
correlation_matrix.to_csv("correlation_matrix.csv",index=True)
#index = ["BT", "BP", "IP", "PLTH/S/2", "NEL", "PGASA"]
df2=pd.read_csv("correlation_matrix.csv",index_col=0)

# Compute condition index
cond_index, max_condition_index = calculate_condition_index(multi_df)
print(cond_index)
print(max_condition_index)
r = calculate_vif1(multi_df)
print(r)
vd_df = calculate_variance_decomposition(multi_df)
print(vd_df)
D = calculate_determinant_of_correlation_matrix(multi_df)
print("Determinant of the Correlation Matrix (D):", D)

   Eigenvalue  Condition Index
0    2.715378         1.000000
1    1.994370         1.166843
2    1.059808         1.600669
3    0.133895         4.503318
4    0.076599         5.953919
5    0.019949        11.666809
11.666808590795178
        Variable        VIF
0  $P_{LH,norm}$   8.869260
1          $B_T$   8.251862
2          $B_P$  19.793986
3          $I_P$  18.450397
4   $n_{e,line}$  14.388756
5      $M_{eff}$   2.709720
               Eigenvalue_1  Eigenvalue_2  Eigenvalue_3  Eigenvalue_4  \
$P_{LH,norm}$      0.771555      0.202797      0.020273      0.002671   
$B_T$              0.231813      0.659761      0.016316      0.053421   
$B_P$              0.019018      0.894961      0.041395      0.017179   
$I_P$              0.724547      0.092466      0.164605      0.001576   
$n_{e,line}$       0.370978      0.596119      0.013454      0.016783   
$M_{eff}$          0.077688      0.000529      0.753898      0.133834   

               Eigenvalue_5  Eigenvalue_6  
$P_{LH,norm}

In [7]:
correlation_matrix

,"$P_{LH,norm}$",$B_T$,$B_P$,$I_P$,"$n_{e,line}$",$M_{eff}$
"$P_{LH,norm}$",1.000000,-0.078095,0.264450,0.818527,-0.166584,-0.350318
$B_T$,-0.078095,1.000000,0.863060,-0.118966,0.868909,0.173681
$B_P$,0.264450,0.863060,1.000000,0.262335,0.775428,0.129850
$I_P$,0.818527,-0.118966,0.262335,1.000000,-0.329181,0.070885
"$n_{e,line}$",-0.166584,0.868909,0.775428,-0.329181,1.000000,0.089726
$M_{eff}$,-0.350318,0.173681,0.129850,0.070885,0.089726,1.000000


In [13]:
correlation_matrix

,$B_T$,$B_P$,$I_P$,"$N_{e,line}$",$M_{eff}$
$B_T$,1.000000,0.863060,-0.118966,0.868909,0.173681
$B_P$,0.863060,1.000000,0.262335,0.775428,0.129850
$I_P$,-0.118966,0.262335,1.000000,-0.329181,0.070885
"$N_{e,line}$",0.868909,0.775428,-0.329181,1.000000,0.089726
$M_{eff}$,0.173681,0.129850,0.070885,0.089726,1.000000


In [16]:
file_path = 'LH_metal_DB_v1.12.csv'  # 替换为你的文件路径
df = pd.read_csv(file_path)
df.loc[:, 'TOK'] = df['TOK'].str.strip()
df['IP']=df['IP'].abs()/1000000
df['BT']=df['BT'].abs()
df['PLTH']=df['PLTH']/1000000
df['NEL']=df['NEL']/1e19


# 指定你想在新 DataFrame 中包含的列
columns_to_include = ['TOK', 'SPLASMA','BT','IP','RGEO','PLTH','NEL','PGASA',"DIVCON"]
new_df = df[df['SELEC2024'] == 1][columns_to_include].copy()
new_df = new_df[new_df['TOK'] == "CMOD"]
mu0 = 4 * np.pi * 1e-7

new_df['BP']=1000000*(mu0 * new_df['IP']) / (new_df['SPLASMA']/(2 * np.pi * new_df['RGEO']))

new_df['PLTH/S/2']=new_df['PLTH']/new_df['SPLASMA']/2


columns_to_include_for_regre = ['PLTH/S/2','BT','BP','IP','NEL']
unique_divcon = new_df['DIVCON'].unique()

# 2. 向原始 DataFrame 添加新列
for divcon_type in unique_divcon:
    new_df[divcon_type] = (new_df['DIVCON'] == divcon_type).astype(int)
multi_df = new_df[columns_to_include_for_regre].copy()
multi_df = multi_df.apply(np.log)
new_columns = ['$P_{LH,norm}$',"$B_T$", "$B_P$", "$I_P$","$n_{e,line}$"]

# 修改列名
multi_df.columns = new_columns
correlation_matrix = multi_df.corr()
correlation_matrix.to_csv("CMOD_correlation_matrix.csv",index=True)
#index = ["BT", "BP", "IP", "PLTH/S/2", "NEL", "PGASA"]
df2=pd.read_csv("CMOD_correlation_matrix.csv",index_col=0)
# Compute condition index
cond_index, max_condition_index = calculate_condition_index(multi_df)
print(cond_index)
print(max_condition_index)
r = calculate_vif1(multi_df)
print(r)
vd_df = calculate_variance_decomposition(multi_df)
print(vd_df)
D = calculate_determinant_of_correlation_matrix(multi_df)
print("Determinant of the Correlation Matrix (D):", D)


   Eigenvalue  Condition Index
0    3.123325         1.000000
1    1.255320         1.577362
2    0.410860         2.757159
3    0.205254         3.900885
4    0.005242        24.408732
24.408732491393287
        Variable        VIF
0  $P_{LH,norm}$   3.242471
1          $B_T$   1.664940
2          $B_P$  99.383723
3          $I_P$  92.350712
4   $n_{e,line}$   2.534720
               Eigenvalue_1  Eigenvalue_2  Eigenvalue_3  Eigenvalue_4  \
$P_{LH,norm}$      0.746335      0.203690      0.010961      0.039015   
$B_T$              0.053008      0.712018      0.216142      0.018826   
$B_P$              0.870358      0.098237      0.025700      0.002717   
$I_P$              0.859257      0.097599      0.039744      0.001155   
$n_{e,line}$       0.563065      0.112633      0.114522      0.209757   

               Eigenvalue_5  
$P_{LH,norm}$  2.956779e-08  
$B_T$          7.343264e-06  
$B_P$          2.987631e-03  
$I_P$          2.244806e-03  
$n_{e,line}$   2.254804e-05  
Determin

In [17]:
file_path = 'LH_metal_DB_v1.12.csv'  # 替换为你的文件路径
df = pd.read_csv(file_path)
df.loc[:, 'TOK'] = df['TOK'].str.strip()
df['IP']=df['IP'].abs()/1000000
df['BT']=df['BT'].abs()
df['PLTH']=df['PLTH']/1000000
df['NEL']=df['NEL']/1e19


# 指定你想在新 DataFrame 中包含的列
columns_to_include = ['TOK', 'SPLASMA','BT','IP','RGEO','PLTH','NEL','PGASA',"DIVCON"]
new_df = df[df['SELEC2024'] == 1][columns_to_include].copy()
new_df = new_df[new_df['TOK'] == "AUG"]
mu0 = 4 * np.pi * 1e-7

new_df['BP']=1000000*(mu0 * new_df['IP']) / (new_df['SPLASMA']/(2 * np.pi * new_df['RGEO']))

new_df['PLTH/S/2']=new_df['PLTH']/new_df['SPLASMA']/2


columns_to_include_for_regre = ['PLTH/S/2','BT','BP','IP','NEL','PGASA']
unique_divcon = new_df['DIVCON'].unique()

# 2. 向原始 DataFrame 添加新列
for divcon_type in unique_divcon:
    new_df[divcon_type] = (new_df['DIVCON'] == divcon_type).astype(int)
multi_df = new_df[columns_to_include_for_regre].copy()
multi_df = multi_df.apply(np.log)
new_columns = ['$P_{LH,norm}$',"$B_T$", "$B_P$", "$I_P$","$n_{e,line}$","$M_{eff}$"]

# 修改列名
multi_df.columns = new_columns
correlation_matrix = multi_df.corr()
correlation_matrix.to_csv("AUG_correlation_matrix.csv",index=True)
#index = ["BT", "BP", "IP", "PLTH/S/2", "NEL", "PGASA"]
df2=pd.read_csv("AUG_correlation_matrix.csv",index_col=0)

# Compute condition index
cond_index, max_condition_index = calculate_condition_index(multi_df)
print(cond_index)
print(max_condition_index)
r = calculate_vif1(multi_df)
print(r)
vd_df = calculate_variance_decomposition(multi_df)
print(vd_df)
D = calculate_determinant_of_correlation_matrix(multi_df)
print("Determinant of the Correlation Matrix (D):", D)

   Eigenvalue  Condition Index
0    3.215868         1.000000
1    1.295785         1.575370
2    0.955556         1.834514
3    0.443347         2.693253
4    0.085486         6.133408
5    0.003959        28.501359
28.50135880580375
        Variable         VIF
0  $P_{LH,norm}$    7.893379
1          $B_T$    1.309998
2          $B_P$  124.346158
3          $I_P$  128.557262
4   $n_{e,line}$    2.193697
5      $M_{eff}$    4.381900
               Eigenvalue_1  Eigenvalue_2  Eigenvalue_3  Eigenvalue_4  \
$P_{LH,norm}$      0.977981      0.003736      0.000110      0.003905   
$B_T$              0.084378      0.032145      0.361606      0.447042   
$B_P$              0.375745      0.556509      0.026353      0.033603   
$I_P$              0.376488      0.564417      0.021476      0.033404   
$n_{e,line}$       0.489162      0.061256      0.389475      0.024152   
$M_{eff}$          0.620411      0.304102      0.010293      0.031152   

               Eigenvalue_5  Eigenvalue_6  
$P_{LH

In [18]:
file_path = 'LH_metal_DB_v1.12.csv'  # 替换为你的文件路径
df = pd.read_csv(file_path)
df.loc[:, 'TOK'] = df['TOK'].str.strip()
df['IP']=df['IP'].abs()/1000000
df['BT']=df['BT'].abs()
df['PLTH']=df['PLTH']/1000000
df['NEL']=df['NEL']/1e19


# 指定你想在新 DataFrame 中包含的列
columns_to_include = ['TOK', 'SPLASMA','BT','IP','RGEO','PLTH','NEL','PGASA',"DIVCON"]
new_df = df[df['SELEC2024'] == 1][columns_to_include].copy()
new_df = new_df[new_df['TOK'] == "JET"]
mu0 = 4 * np.pi * 1e-7

new_df['BP']=1000000*(mu0 * new_df['IP']) / (new_df['SPLASMA']/(2 * np.pi * new_df['RGEO']))

new_df['PLTH/S/2']=new_df['PLTH']/new_df['SPLASMA']/2


columns_to_include_for_regre = ['PLTH/S/2','BT','BP','IP','NEL','PGASA']
unique_divcon = new_df['DIVCON'].unique()

# 2. 向原始 DataFrame 添加新列
for divcon_type in unique_divcon:
    new_df[divcon_type] = (new_df['DIVCON'] == divcon_type).astype(int)
multi_df = new_df[columns_to_include_for_regre].copy()
multi_df = multi_df.apply(np.log)
new_columns = ['$P_{LH,norm}$',"$B_T$", "$B_P$", "$I_P$","$n_{e,line}$","$M_{eff}$"]

# 修改列名
multi_df.columns = new_columns
correlation_matrix = multi_df.corr()
correlation_matrix.to_csv("JET_correlation_matrix.csv",index=True)
#index = ["BT", "BP", "IP", "PLTH/S/2", "NEL", "PGASA"]
df2=pd.read_csv("JET_correlation_matrix.csv",index_col=0)
# Compute condition index
cond_index, max_condition_index = calculate_condition_index(multi_df)
print(cond_index)
print(max_condition_index)
r = calculate_vif1(multi_df)
print(r)
vd_df = calculate_variance_decomposition(multi_df)
print(vd_df)
D = calculate_determinant_of_correlation_matrix(multi_df)
print("Determinant of the Correlation Matrix (D):", D)


   Eigenvalue  Condition Index
0    3.523652         1.000000
1    1.554996         1.505330
2    0.607734         2.407908
3    0.178761         4.439767
4    0.133543         5.136728
5    0.001315        51.765801
51.76580146961261
        Variable         VIF
0  $P_{LH,norm}$    3.991663
1          $B_T$    4.029932
2          $B_P$  379.240702
3          $I_P$  383.531193
4   $n_{e,line}$    2.202774
5      $M_{eff}$    3.147347
               Eigenvalue_1  Eigenvalue_2  Eigenvalue_3  Eigenvalue_4  \
$P_{LH,norm}$      0.972856      0.015996      0.001224      0.009730   
$B_T$              0.168438      0.607539      0.124954      0.013266   
$B_P$              0.315101      0.532921      0.103010      0.002445   
$I_P$              0.307207      0.545935      0.098273      0.002973   
$n_{e,line}$       0.299383      0.339636      0.293851      0.067085   
$M_{eff}$          0.246821      0.626423      0.033905      0.092633   

               Eigenvalue_5  Eigenvalue_6  
$P_{LH